# NeuroBridge-S4 Graph Learning — Phase 5: Hazard-Aware Graph Embeddings and Similarity Mapping

Phase 5 maps each biological adaptation graph into a shared **graph-feature space** and adds an
**HRP five-hazard context layer**. This lets pseudo-crew participants be compared not only by
biological graph structure, but also by how their activated domains map onto NASA's five human
spaceflight hazard categories.

> **Core positioning**
>
> NeuroBridge-S4 connects individual biological adaptation patterns to NASA's five human
> spaceflight hazard categories without claiming actual exposure, diagnosis, or causal proof.


## 1. Plain-language purpose

Phase 5 maps each biological adaptation graph into a graph-feature space and adds an HRP
five-hazard context layer. This allows pseudo-crew participants to be compared not only by
biological graph structure, but also by how their activated domains map onto NASA's five human
spaceflight hazard categories.

In practice, Phase 5:

1. adds a hazard-context layer mapping biological domains to the five HRP hazards;
2. computes a **hazard relevance score** per participant per hazard;
3. adds those hazard-context features to the Phase 4 graph-feature matrix;
4. compares participants using **cosine similarity** and **Euclidean distance**;
5. produces a transparent **PCA** view of the feature space;
6. writes reviewer-friendly tables, figures, and a plain-language report.


## 2. Why the HRP hazard layer matters

NASA HRP organizes human spaceflight challenges around five major hazards:

- **Space Radiation**
- **Isolation and Confinement**
- **Distance from Earth**
- **Gravity Fields**
- **Hostile / Closed Environments**

NeuroBridge-S4 should speak this HRP language by translating biological graph patterns into
hazard-context relevance scores.

> A **hazard relevance score** is not a measured exposure score and not a health risk score.
> It is a transparent mapping from activated biological domains to HRP hazard categories,
> designed to help reviewers interpret which spaceflight hazard contexts may be most relevant
> for closer monitoring.

**Distance from Earth** is treated differently from the others. It is not primarily a biological
domain; it increases the operational value of autonomous, interpretable monitoring and early
signal triage when real-time Earth support is limited.


## 3. Scientific guardrails

- This is **not** actual Artemis II data.
- These are **proxy-data-derived** graph features.
- Hazard relevance does **not** mean actual hazard exposure.
- Hazard relevance does **not** mean disease, risk, or diagnosis.
- Similarity does **not** mean the same health state.
- PCA is a **visualization** of graph-feature space, not a classifier.
- No clinical or psychological conditions are inferred.

> *Hazard-context mapping: biological domains and graph patterns are mapped to NASA HRP's five
> human spaceflight hazard categories as interpretation context, not as exposure measurement or
> causal attribution.*


## Setup

In [1]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')  # non-interactive backend — safe for notebook/CI
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from neurobridge_graph.hazard_mapping import (
    get_default_hazard_domain_mapping,
    export_hazard_domain_mapping,
    compute_hazard_relevance_scores,
    compute_hazard_coverage,
    HAZARD_CANONICAL,
    HAZARD_DISPLAY_NAMES,
    CORE_POSITIONING_SENTENCE,
)
from neurobridge_graph.embeddings import (
    load_phase4_feature_tables,
    build_hazard_aware_feature_matrix,
    select_numeric_features,
    scale_feature_matrix,
    compute_pca_embedding,
)
from neurobridge_graph.similarity import (
    compute_cosine_similarity_matrix,
    compute_euclidean_distance_matrix,
    summarize_pairwise_similarity,
    identify_most_similar_pair,
    identify_most_distinct_subject,
)

RESULTS = repo_root / 'results'
TABLES = RESULTS / 'tables'
FIGURES = RESULTS / 'figures'
REPORTS = RESULTS / 'reports'
for d in (TABLES, FIGURES, REPORTS):
    d.mkdir(parents=True, exist_ok=True)

print('Setup complete.')
print(CORE_POSITIONING_SENTENCE)


Setup complete.
NeuroBridge-S4 connects individual biological adaptation patterns to NASA's five human spaceflight hazard categories without claiming actual exposure, diagnosis, or causal proof.


## 5. Load Phase 4 feature tables

Phase 5 builds on the Phase 4 feature tables. Graph-level and node-level features are **required**;
subgraph and edge-level features are **recommended**.


In [2]:
tables = load_phase4_feature_tables(RESULTS)
print('Loaded Phase 4 tables:', list(tables.keys()))

graph_level = tables['graph_level_features']
node_level = tables['node_level_features']
subgraph = tables.get('subgraph_features')
edge_level = tables.get('edge_level_features')

expected = [
    ('graph_level_features.csv', 'graph_level_features', 'required'),
    ('node_level_features.csv',  'node_level_features',  'required'),
    ('subgraph_features.csv',    'subgraph_features',    'recommended'),
    ('edge_level_features.csv',  'edge_level_features',  'recommended'),
]
readiness_rows = []
for fname, key, req in expected:
    df = tables.get(key)
    readiness_rows.append({
        'file': fname,
        'status': 'present' if df is not None else 'missing',
        'rows': int(len(df)) if df is not None else 0,
        'columns': int(df.shape[1]) if df is not None else 0,
        'required_or_optional': req,
    })
input_readiness = pd.DataFrame(readiness_rows)
input_readiness.to_csv(TABLES / 'phase5_input_readiness_check.csv', index=False)
print('Saved -> results/tables/phase5_input_readiness_check.csv')
input_readiness


Loaded Phase 4 tables: ['graph_level_features', 'node_level_features', 'edge_level_features', 'subgraph_features']
Saved -> results/tables/phase5_input_readiness_check.csv


,file,status,rows,columns,required_or_optional
0,graph_level_features.csv,present,4,18,required
1,node_level_features.csv,present,24,11,required
2,subgraph_features.csv,present,20,11,recommended
3,edge_level_features.csv,present,17,8,recommended


## 6. Build the HRP hazard-domain mapping

The mapping below is a **conceptual HRP relevance mapping**, not exposure measurement. Each row
links a biological domain to a hazard with a relevance weight in `[0, 1]`.


In [3]:
hazard_mapping = get_default_hazard_domain_mapping()
export_hazard_domain_mapping(TABLES / 'hazard_domain_mapping.csv', hazard_mapping)
print('Saved -> results/tables/hazard_domain_mapping.csv')
print(f'{hazard_mapping.shape[0]} domain-hazard relevance links across '
      f'{hazard_mapping["hazard"].nunique()} hazards.')

hazard_mapping[['hazard_display', 'domain', 'weight', 'interpretation']]


Saved -> results/tables/hazard_domain_mapping.csv
28 domain-hazard relevance links across 5 hazards.


,hazard_display,domain,weight,interpretation
0,Space Radiation,inflammation / immune-adjacent status,0.8,'inflammation / immune-adjacent status' is con...
1,Space Radiation,hematologic / oxygen-carrying capacity,0.7,'hematologic / oxygen-carrying capacity' is co...
2,Space Radiation,cardiovascular regulation,0.5,'cardiovascular regulation' is conceptually re...
3,Space Radiation,recovery-related markers,0.6,'recovery-related markers' is conceptually rel...
4,Space Radiation,cognitive load,0.4,'cognitive load' is conceptually relevant to S...
5,Isolation and Confinement,sleep / circadian regulation,0.9,'sleep / circadian regulation' is conceptually...
6,Isolation and Confinement,autonomic regulation,0.8,'autonomic regulation' is conceptually relevan...
7,Isolation and Confinement,emotional regulation,0.9,'emotional regulation' is conceptually relevan...
8,Isolation and Confinement,cognitive load,0.8,'cognitive load' is conceptually relevant to I...
9,Isolation and Confinement,inflammation / immune-adjacent status,0.4,'inflammation / immune-adjacent status' is con...


## 7. Compute hazard relevance scores

For each subject and hazard:

```
hazard_relevance_score = sum(domain_activation * hazard_domain_weight) / sum(hazard_domain_weight)
```

computed over the mapped domains that are actually available in the proxy dataset.


In [4]:
hazard_scores = compute_hazard_relevance_scores(node_level, hazard_mapping)
hazard_coverage = compute_hazard_coverage(node_level, hazard_mapping)

hazard_scores.to_csv(TABLES / 'hazard_relevance_scores.csv', index=False)
hazard_coverage.to_csv(TABLES / 'hazard_coverage_report.csv', index=False)
print('Saved -> results/tables/hazard_relevance_scores.csv')
print('Saved -> results/tables/hazard_coverage_report.csv')

hazard_scores[['subject_id', 'hazard_display', 'hazard_relevance_score',
               'coverage_fraction', 'top_contributing_domain', 'interpretation']]


Saved -> results/tables/hazard_relevance_scores.csv
Saved -> results/tables/hazard_coverage_report.csv


,subject_id,hazard_display,hazard_relevance_score,coverage_fraction,top_contributing_domain,interpretation
0,Crew 100987,Space Radiation,0.83978,0.80000,cardiovascular regulation,Space Radiation: mild hazard-context relevance...
1,Crew 100987,Isolation and Confinement,0.48617,0.16667,inflammation / immune-adjacent status,Isolation and Confinement: low hazard-context ...
2,Crew 100987,Distance from Earth,1.88191,0.20000,cardiovascular regulation,Distance from Earth: high hazard-context relev...
3,Crew 100987,Gravity Fields,0.93934,0.83333,cardiovascular regulation,Gravity Fields: mild hazard-context relevance ...
4,Crew 100987,Hostile / Closed Environments,0.61357,0.50000,recovery-related markers,Hostile / Closed Environments: low hazard-cont...
5,Crew 94465,Space Radiation,0.69027,0.80000,recovery-related markers,Space Radiation: low hazard-context relevance ...
6,Crew 94465,Isolation and Confinement,0.49645,0.16667,inflammation / immune-adjacent status,Isolation and Confinement: low hazard-context ...
7,Crew 94465,Distance from Earth,0.94861,0.20000,cardiovascular regulation,Distance from Earth: mild hazard-context relev...
8,Crew 94465,Gravity Fields,0.94285,0.83333,body composition / physical status,Gravity Fields: mild hazard-context relevance ...
9,Crew 94465,Hostile / Closed Environments,0.68961,0.50000,recovery-related markers,Hostile / Closed Environments: low hazard-cont...


### Domain coverage limitations

> The current NHANES-derived proxy demonstration has limited coverage for some HRP hazard
> categories, especially isolation/confinement and distance-from-Earth autonomy contexts. This
> limitation is informative: it shows which additional Artemis II data streams (sleep/circadian,
> autonomic, emotional, and cognitive-load measures) would strengthen hazard-context interpretation.


In [5]:
hazard_coverage[['hazard_display', 'available_domain_count', 'expected_domain_count',
                 'coverage_fraction', 'missing_domains']]


,hazard_display,available_domain_count,expected_domain_count,coverage_fraction,missing_domains
0,Space Radiation,4,5,0.80000,cognitive load
1,Isolation and Confinement,1,6,0.16667,sleep / circadian regulation; autonomic regula...
2,Distance from Earth,1,5,0.20000,recovery capacity; cognitive load; emotional r...
3,Gravity Fields,5,6,0.83333,autonomic regulation
4,Hostile / Closed Environments,3,6,0.50000,sleep / circadian regulation; autonomic regula...


## 8. Visualize hazard relevance

> This heatmap does **not** show actual exposure to hazards. It shows which biological domain
> activations map most strongly to HRP hazard contexts under the current proxy-data schema.


In [6]:
def annotated_heatmap(ax, matrix, row_labels, col_labels, cmap, title,
                      cbar_label, fmt='{:.2f}', vmin=None, vmax=None):
    data = np.array(matrix, dtype=float)
    im = ax.imshow(data, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=9)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data[i, j]
            txt = 'n/a' if np.isnan(val) else fmt.format(val)
            ax.text(j, i, txt, ha='center', va='center', fontsize=8, color='#222')
    ax.set_title(title, fontsize=11, pad=10)
    cbar = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(cbar_label, fontsize=8)
    return im


subjects = list(graph_level['subject_id'].astype(str))
hazard_order = HAZARD_CANONICAL
hazard_labels = [HAZARD_DISPLAY_NAMES[h] for h in hazard_order]

rel_pivot = hazard_scores.pivot(index='subject_id', columns='hazard',
                                values='hazard_relevance_score').reindex(
    index=subjects, columns=hazard_order)

fig, ax = plt.subplots(figsize=(9, 4.5))
annotated_heatmap(ax, rel_pivot.values, subjects, hazard_labels, 'YlOrRd',
                  'Hazard relevance scores (pseudo-crew x HRP hazards)',
                  'Hazard relevance score', vmin=0)
fig.text(0.5, -0.04,
         'Interpretation context only. Not exposure measurement, diagnosis, or causal proof.',
         ha='center', fontsize=8, color='#555', style='italic')
plt.tight_layout()
fig.savefig(FIGURES / 'phase5_hazard_relevance_heatmap.png', dpi=130, bbox_inches='tight')
plt.close(fig)
print('Saved -> results/figures/phase5_hazard_relevance_heatmap.png')


Saved -> results/figures/phase5_hazard_relevance_heatmap.png


In [7]:
cov_pivot = hazard_coverage.set_index('hazard').reindex(hazard_order)
cov_values = cov_pivot[['coverage_fraction']].values
cov_annot = [[f"{int(r['available_domain_count'])}/{int(r['expected_domain_count'])}"]
             for _, r in cov_pivot.iterrows()]

fig, ax = plt.subplots(figsize=(5.5, 4.2))
im = ax.imshow(cov_values, cmap='Greens', aspect='auto', vmin=0, vmax=1)
ax.set_xticks([0]); ax.set_xticklabels(['domain coverage'], fontsize=9)
ax.set_yticks(range(len(hazard_labels))); ax.set_yticklabels(hazard_labels, fontsize=9)
for i in range(len(hazard_labels)):
    ax.text(0, i, f"{cov_values[i,0]*100:.0f}%  ({cov_annot[i][0]})",
            ha='center', va='center', fontsize=8, color='#222')
ax.set_title('HRP hazard domain coverage in proxy data', fontsize=11, pad=10)
cbar = fig.colorbar(im, ax=ax, fraction=0.06, pad=0.06)
cbar.set_label('coverage fraction', fontsize=8)
fig.text(0.5, -0.04,
         'Low coverage indicates missing proxy data streams, not absence of hazard relevance.',
         ha='center', fontsize=8, color='#555', style='italic')
plt.tight_layout()
fig.savefig(FIGURES / 'phase5_hazard_coverage_heatmap.png', dpi=130, bbox_inches='tight')
plt.close(fig)
print('Saved -> results/figures/phase5_hazard_coverage_heatmap.png')


Saved -> results/figures/phase5_hazard_coverage_heatmap.png


## 9. Build the hazard-aware feature matrix

The feature matrix combines five families of features:

- **graph activation features** (mean / max / total node activation, active-domain counts);
- **graph connectivity features** (density, edge counts);
- **subgraph features** (template activation means);
- **BACI context** (`baci_score`);
- **HRP hazard relevance features** (`hazard_relevance__*`).


In [8]:
feature_matrix = build_hazard_aware_feature_matrix(
    graph_level_features=graph_level,
    node_level_features=node_level,
    subgraph_features=subgraph,
    hazard_relevance_scores=hazard_scores,
    include_baci=True,
)
feature_matrix.to_csv(TABLES / 'phase5_hazard_aware_feature_matrix.csv', index=False)
print('Saved -> results/tables/phase5_hazard_aware_feature_matrix.csv')

X, feature_names = select_numeric_features(feature_matrix)
X_scaled, scaler = scale_feature_matrix(X, method='standard')

scaled_matrix = X_scaled.copy()
scaled_matrix.insert(0, 'subject_id', list(feature_matrix['subject_id']))
scaled_matrix.to_csv(TABLES / 'phase5_hazard_aware_scaled_feature_matrix.csv', index=False)
print('Saved -> results/tables/phase5_hazard_aware_scaled_feature_matrix.csv')

hazard_feats = [c for c in feature_names if c.startswith('hazard_relevance__')]
print(f'\nTotal numeric features: {len(feature_names)}')
print(f'Hazard relevance features: {len(hazard_feats)}')
feature_matrix


Saved -> results/tables/phase5_hazard_aware_feature_matrix.csv
Saved -> results/tables/phase5_hazard_aware_scaled_feature_matrix.csv

Total numeric features: 22
Hazard relevance features: 5


,subject_id,mean_node_activation,median_node_activation,max_node_activation,total_node_activation,n_active_domains,active_domain_fraction,graph_density,mean_edge_weight,max_edge_weight,...,baci_score,subgraph_activation_mean__cardiometabolic,subgraph_activation_mean__cognitive_emotional_recovery,subgraph_activation_mean__hematologic_cardiovascular,subgraph_activation_mean__immune_metabolic,hazard_relevance__distance_from_earth,hazard_relevance__gravity_fields,hazard_relevance__hostile_closed_environments,hazard_relevance__isolation_and_confinement,hazard_relevance__space_radiation
0,Crew 100987,0.84097,0.68918,1.88191,5.04580,1,0.16667,0.26667,1.00000,1.00000,...,28.6,1.07985,0.70515,1.06733,0.62683,1.88191,0.93934,0.61357,0.48617,0.83978
1,Crew 94465,0.83948,0.83517,1.32253,5.03688,1,0.16667,0.26667,1.00000,1.00000,...,22.1,1.03544,0.80638,0.79424,0.71267,0.94861,0.94285,0.68961,0.49645,0.69027
2,Crew 97774,1.05170,0.94380,2.10729,6.31018,2,0.33333,0.33333,1.19746,1.98728,...,41.5,0.94557,0.94380,1.17834,0.61724,0.48392,1.17727,0.62067,0.42240,1.00818
3,Crew 99813,0.69729,0.83698,1.15790,4.18376,2,0.33333,0.26667,1.00000,1.00000,...,31.5,0.51546,0.83698,0.57750,1.03111,0.19356,0.57338,1.02180,1.09845,0.75734


## 10. Compute similarity and distance

- **cosine similarity** = similarity of the graph-feature *pattern*;
- **Euclidean distance** = absolute separation in scaled feature space.

Both are structural comparisons of graph-feature vectors, **not** diagnosis.


In [9]:
subject_ids = list(feature_matrix['subject_id'].astype(str))

similarity_matrix = compute_cosine_similarity_matrix(X_scaled, subject_ids)
distance_matrix = compute_euclidean_distance_matrix(X_scaled, subject_ids)
similarity_summary = summarize_pairwise_similarity(similarity_matrix, distance_matrix)

similarity_matrix.to_csv(TABLES / 'graph_similarity_matrix.csv')
distance_matrix.to_csv(TABLES / 'graph_distance_matrix.csv')
similarity_summary.to_csv(TABLES / 'phase5_similarity_summary.csv', index=False)
print('Saved -> results/tables/graph_similarity_matrix.csv')
print('Saved -> results/tables/graph_distance_matrix.csv')
print('Saved -> results/tables/phase5_similarity_summary.csv')

most_similar = identify_most_similar_pair(similarity_summary)
most_distinct = identify_most_distinct_subject(distance_matrix)
print('\nMost similar pair :', most_similar)
print('Most distinct     :', most_distinct)
similarity_summary


Saved -> results/tables/graph_similarity_matrix.csv
Saved -> results/tables/graph_distance_matrix.csv
Saved -> results/tables/phase5_similarity_summary.csv

Most similar pair : {'subject_a': 'Crew 100987', 'subject_b': 'Crew 94465', 'cosine_similarity': 0.34812, 'euclidean_distance': 3.81684}
Most distinct     : {'subject_id': 'Crew 97774', 'mean_distance_to_others': 8.62392}


,subject_a,subject_b,cosine_similarity,euclidean_distance
0,Crew 100987,Crew 94465,0.34812,3.81684
1,Crew 94465,Crew 99813,-0.02917,6.00354
2,Crew 100987,Crew 97774,-0.33218,7.96081
3,Crew 100987,Crew 99813,-0.52708,7.77448
4,Crew 97774,Crew 99813,-0.52953,9.72074
5,Crew 94465,Crew 97774,-0.67974,8.19020


## 11. Visualize similarity and distance

In [10]:
fig, ax = plt.subplots(figsize=(6, 5))
annotated_heatmap(ax, similarity_matrix.values, subject_ids, subject_ids, 'Blues',
                  'How similar are pseudo-crew biological adaptation\ngraphs in hazard-aware feature space?',
                  'cosine similarity', vmin=-1, vmax=1)
plt.tight_layout()
fig.savefig(FIGURES / 'phase5_hazard_aware_similarity_heatmap.png', dpi=130, bbox_inches='tight')
plt.close(fig)
print('Saved -> results/figures/phase5_hazard_aware_similarity_heatmap.png')

fig, ax = plt.subplots(figsize=(6, 5))
annotated_heatmap(ax, distance_matrix.values, subject_ids, subject_ids, 'Purples',
                  'How far apart are pseudo-crew graphs\nin hazard-aware feature space?',
                  'Euclidean distance', vmin=0)
plt.tight_layout()
fig.savefig(FIGURES / 'phase5_hazard_aware_distance_heatmap.png', dpi=130, bbox_inches='tight')
plt.close(fig)
print('Saved -> results/figures/phase5_hazard_aware_distance_heatmap.png')


Saved -> results/figures/phase5_hazard_aware_similarity_heatmap.png
Saved -> results/figures/phase5_hazard_aware_distance_heatmap.png


## 12. PCA embedding

PCA is used as a **transparent visualization** of graph-feature space, not as a classifier.


In [11]:
embedding, pca = compute_pca_embedding(X_scaled, subject_ids, n_components=2)
embedding.to_csv(TABLES / 'graph_embeddings.csv', index=False)
print('Saved -> results/tables/graph_embeddings.csv')

if pca is not None and 'PC1' in embedding.columns and 'PC2' in embedding.columns:
    evr = pca.explained_variance_ratio_
    fig, ax = plt.subplots(figsize=(7, 5.5))
    ax.scatter(embedding['PC1'], embedding['PC2'], s=120, c='#2471a3',
               edgecolor='#13395e', zorder=3)
    for _, row in embedding.iterrows():
        ax.annotate(str(row['subject_id']), (row['PC1'], row['PC2']),
                    textcoords='offset points', xytext=(8, 5), fontsize=9)
    ax.axhline(0, color='#ccc', lw=0.8, zorder=1)
    ax.axvline(0, color='#ccc', lw=0.8, zorder=1)
    ax.set_xlabel(f'PC1 ({evr[0]*100:.1f}% variance)', fontsize=10)
    ax.set_ylabel(f'PC2 ({evr[1]*100:.1f}% variance)' if len(evr) > 1 else 'PC2', fontsize=10)
    ax.set_title('Pseudo-crew graphs in hazard-aware feature space (PCA)', fontsize=11)
    fig.text(0.5, -0.02,
             'Because this demonstration has only four pseudo-crew participants, the PCA plot is '
             'used as an\ninterpretable visualization of feature-space separation, not as a '
             'validated manifold or clustering result.',
             ha='center', fontsize=8, color='#555', style='italic')
    plt.tight_layout()
    fig.savefig(FIGURES / 'phase5_hazard_aware_pca_embedding.png', dpi=130, bbox_inches='tight')
    plt.close(fig)
    print('Saved -> results/figures/phase5_hazard_aware_pca_embedding.png')
else:
    print('PCA skipped (insufficient samples/features).')


Saved -> results/tables/graph_embeddings.csv


Saved -> results/figures/phase5_hazard_aware_pca_embedding.png


> Because this demonstration has only four pseudo-crew participants, the PCA plot is used as an
> interpretable visualization of feature-space separation, not as a validated manifold or
> clustering result.


## 13. Feature contribution plot

PCA loadings show which features most strongly drive separation along PC1 and PC2. Hazard
relevance features are highlighted to show whether the HRP layer contributes meaningfully.


In [12]:
if pca is not None and pca.components_.shape[0] >= 1:
    n_pc = min(2, pca.components_.shape[0])
    fig, axes = plt.subplots(1, n_pc, figsize=(6.5 * n_pc, 5))
    if n_pc == 1:
        axes = [axes]
    for pc_idx in range(n_pc):
        loadings = pd.Series(pca.components_[pc_idx], index=feature_names)
        top = loadings.reindex(loadings.abs().sort_values(ascending=False).index)[:8][::-1]
        colors = ['#cf6f6a' if f.startswith('hazard_relevance__') else '#2471a3'
                  for f in top.index]
        axes[pc_idx].barh(range(len(top)), top.values, color=colors)
        axes[pc_idx].set_yticks(range(len(top)))
        axes[pc_idx].set_yticklabels(top.index, fontsize=8)
        axes[pc_idx].axvline(0, color='#888', lw=0.8)
        axes[pc_idx].set_title(f'Top contributors to PC{pc_idx + 1}', fontsize=11)
        axes[pc_idx].set_xlabel('PCA loading', fontsize=9)
    fig.suptitle('Feature contributions to hazard-aware PCA (red = HRP hazard relevance feature)',
                 fontsize=11, y=1.02)
    plt.tight_layout()
    fig.savefig(FIGURES / 'phase5_hazard_aware_feature_contribution_plot.png',
                dpi=130, bbox_inches='tight')
    plt.close(fig)
    print('Saved -> results/figures/phase5_hazard_aware_feature_contribution_plot.png')

    hazard_in_top = False
    for pc_idx in range(n_pc):
        loadings = pd.Series(pca.components_[pc_idx], index=feature_names)
        top_feats = loadings.abs().sort_values(ascending=False).index[:8]
        if any(f.startswith('hazard_relevance__') for f in top_feats):
            hazard_in_top = True
    print('Hazard relevance features among top PCA contributors:', hazard_in_top)
else:
    print('Feature contribution plot skipped (no PCA available).')
    hazard_in_top = False


Saved -> results/figures/phase5_hazard_aware_feature_contribution_plot.png
Hazard relevance features among top PCA contributors: True


## 14. Plain-language Phase 5 report

In [13]:
lines = []
lines.append('NeuroBridge-S4 Graph Learning — Phase 5 Report')
lines.append('Hazard-Aware Graph Embeddings and Similarity Mapping')
lines.append('=' * 64)
lines.append('')
lines.append(CORE_POSITIONING_SENTENCE)
lines.append('')
lines.append('Overview')
lines.append('-' * 64)
lines.append('Phase 5 maps each biological adaptation graph into a hazard-aware graph-feature')
lines.append('space and adds NASA HRP five-hazard context relevance scores. It does not assign')
lines.append('clinical labels. It maps biological adaptation graphs into a hazard-aware feature')
lines.append('space so that reviewers can inspect structural similarity and HRP hazard-context')
lines.append('relevance.')
lines.append('')
lines.append('Analysis summary')
lines.append('-' * 64)
lines.append(f'Participants analyzed       : {len(subject_ids)}')
lines.append(f'Total graph features used   : {len(feature_names)}')
lines.append(f'Hazard relevance features   : {len(hazard_feats)}')
n_subgraph_feats = len([c for c in feature_names if c.startswith("subgraph_activation_mean__")])
lines.append(f'Subgraph features used      : {n_subgraph_feats}')
lines.append('')
lines.append('Strongest hazard-context pattern per participant')
lines.append('-' * 64)
for sid in subject_ids:
    sub = hazard_scores[hazard_scores['subject_id'] == sid].dropna(subset=['hazard_relevance_score'])
    if len(sub):
        top = sub.sort_values('hazard_relevance_score', ascending=False).iloc[0]
        lines.append(f'  {sid}: {top["hazard_display"]} '
                     f'(relevance {top["hazard_relevance_score"]:.2f}, '
                     f'coverage {top["coverage_fraction"]*100:.0f}%)')
    else:
        lines.append(f'  {sid}: no hazard relevance computable (no mapped domains).')
lines.append('')
lines.append('Most similar participant pair')
lines.append('-' * 64)
if most_similar.get('subject_a') is not None:
    lines.append(f'  {most_similar["subject_a"]} <-> {most_similar["subject_b"]} '
                 f'(cosine similarity {most_similar["cosine_similarity"]:.3f}, '
                 f'Euclidean distance {most_similar["euclidean_distance"]:.3f})')
else:
    lines.append('  Not available (fewer than 2 participants).')
lines.append('')
lines.append('Most distinct participant')
lines.append('-' * 64)
lines.append(f'  {most_distinct.get("subject_id")} '
             f'(mean distance to others {most_distinct.get("mean_distance_to_others")})')
lines.append('')
lines.append('Top features contributing to separation')
lines.append('-' * 64)
if pca is not None:
    pc1 = pd.Series(pca.components_[0], index=feature_names)
    for f in pc1.reindex(pc1.abs().sort_values(ascending=False).index)[:6].index:
        lines.append(f'  PC1: {f} (loading {pc1[f]:+.3f})')
lines.append('')
lines.append('Domain coverage limitations')
lines.append('-' * 64)
for _, r in hazard_coverage.iterrows():
    lines.append(f'  {r["hazard_display"]}: {r["available_domain_count"]}/'
                 f'{r["expected_domain_count"]} domains '
                 f'({r["coverage_fraction"]*100:.0f}%).')
lines.append('  The current NHANES-derived proxy has limited coverage for isolation/confinement')
lines.append('  and distance-from-Earth autonomy contexts. Additional Artemis II data streams')
lines.append('  (sleep/circadian, autonomic, emotional, cognitive-load) would strengthen coverage.')
lines.append('')
lines.append('Interpretation guardrail')
lines.append('-' * 64)
lines.append('  Hazard relevance is not exposure measurement, risk scoring, diagnosis, or causal')
lines.append('  attribution. Similarity is structural, not clinical. PCA is a visualization, not a')
lines.append('  classifier. No clinical or psychological conditions are inferred.')
lines.append('')
lines.append('Next step')
lines.append('-' * 64)
lines.append('  Phase 6 — reference-relative graph novelty detection: learn a reference graph space')
lines.append('  from many proxy participants and interpret the n=4 pseudo-crew relative to it.')
lines.append('')

report_text = '\n'.join(lines)
report_path = REPORTS / 'phase5_hazard_aware_similarity_report.txt'
report_path.write_text(report_text, encoding='utf-8')
print(f'Saved -> {report_path.relative_to(repo_root)}')
print()
print(report_text)


Saved -> results/reports/phase5_hazard_aware_similarity_report.txt

NeuroBridge-S4 Graph Learning — Phase 5 Report
Hazard-Aware Graph Embeddings and Similarity Mapping

NeuroBridge-S4 connects individual biological adaptation patterns to NASA's five human spaceflight hazard categories without claiming actual exposure, diagnosis, or causal proof.

Overview
----------------------------------------------------------------
Phase 5 maps each biological adaptation graph into a hazard-aware graph-feature
space and adds NASA HRP five-hazard context relevance scores. It does not assign
clinical labels. It maps biological adaptation graphs into a hazard-aware feature
space so that reviewers can inspect structural similarity and HRP hazard-context
relevance.

Analysis summary
----------------------------------------------------------------
Participants analyzed       : 4
Total graph features used   : 22
Hazard relevance features   : 5
Subgraph features used      : 4

Strongest hazard-context patt

## 15. Extension to a reference graph population

> The current demonstration may use only pseudo-crew graphs. The scientifically preferred
> extension is to build a larger reference graph population from real proxy participants and
> place the n=4 pseudo-crew inside that reference graph space.

> We should not train on n=4. We should learn the reference graph space from many proxy
> participants and interpret the n=4 pseudo-crew relative to that space.


## 16. Conclusion

Phase 5 translates graph structure into HRP-aligned hazard-aware similarity maps. Each pseudo-crew
participant is now represented in a shared graph-feature space that includes biological graph
features and HRP five-hazard relevance context.

Phase 6 will build on this by computing **reference-relative graph novelty** scores — interpreting
each participant's adaptation graph relative to a larger reference graph population rather than
training on a tiny sample.
